# 🕷️ Scraping CNaPS — Actualités avec pagination (Crawl4AI)
Scrape dynamiquement tous les articles de la page **Actualités** du site CNaPS,
en gérant automatiquement la pagination.

**Prérequis :** Container Docker Crawl4AI actif sur le port `11235`
```powershell
docker run -d -p 11235:11235 --name crawl4ai --shm-size=3g --dns 8.8.8.8 unclecode/crawl4ai:0.8.6
```

## Cellule 1 — Installation des dépendances

In [1]:
!pip install -q tqdm requests pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Johns\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Cellule 2 — Configuration & fonction helper

In [2]:
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm
from typing import Optional

# ── Configuration ────────────────────────────────────────────────────────────
CRAWL4AI_API        = "http://localhost:11235/crawl"
CNAPS_ACTUALITES    = "https://www.cnaps.mg/fr/categorie/actualites"
CNAPS_BASE_URL      = "https://www.cnaps.mg"

# Délai entre chaque batch pour ne pas surcharger le site (secondes)
DELAI_ENTRE_REQUETES = 1.5

# ── Helper : appel à l'API Crawl4AI ──────────────────────────────────────────
def crawl_urls(
    urls: list[str],
    attendre_js: bool = True,
    timeout: int = 30
) -> Optional[list[dict]]:
    """
    Envoie une liste d'URLs au container Docker Crawl4AI et retourne
    la liste des résultats bruts (un dict par URL).

    Args:
        urls: Liste des URLs à crawler.
        attendre_js: Si True, attend le rendu JS complet avant extraction.
        timeout: Délai max (secondes) pour la requête HTTP vers l'API.

    Returns:
        Liste de dicts crawl4ai, ou None en cas d'erreur réseau.
    """
    payload = {
        "urls": urls,
        "magic": True,         # Anti-bot automatique
        "readability": True,   # Nettoyage du HTML → texte lisible
        "wait_until": "domcontentloaded",
    }
    # Pour les pages dynamiques (SPA/JS), on attend que les cards soient visibles
    if attendre_js:
        payload["js_code"] = "window.scrollTo(0, document.body.scrollHeight);"
        payload["wait_for"] = "css:article, .post, .actualite, .entry-title, h2 a"

    try:
        resp = requests.post(
            CRAWL4AI_API,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=timeout,
        )
        resp.raise_for_status()
        data = resp.json()
        return data.get("results", [])
    except requests.exceptions.HTTPError as e:
        print(f"❌ Erreur HTTP {resp.status_code} : {e}")
    except requests.exceptions.ConnectionError:
        print("❌ Impossible de joindre le container Docker. Est-il démarré ?")
    except requests.exceptions.Timeout:
        print("❌ Timeout — le container met trop de temps à répondre.")
    return None


print("✅ Configuration chargée.")
print(f"   API Crawl4AI : {CRAWL4AI_API}")
print(f"   Page cible   : {CNAPS_ACTUALITES}")

✅ Configuration chargée.
   API Crawl4AI : http://localhost:11235/crawl
   Page cible   : https://www.cnaps.mg/fr/categorie/actualites


## Cellule 3 — Découverte de toutes les pages + collecte des URLs d'articles

In [4]:
def _extraire_texte_markdown(resultat: dict) -> str:
    """
    Extrait le texte markdown depuis un résultat crawl4ai.

    Dans crawl4ai >= 0.3, le champ "markdown" est un dict (MarkdownGenerationResult)
    et non une string. On tente les clés dans l'ordre de préférence.

    Args:
        resultat: Dict résultat crawl4ai brut.

    Returns:
        Texte markdown sous forme de string (jamais None).
    """
    raw = resultat.get("markdown", "")

    # crawl4ai >= 0.3 : markdown est un MarkdownGenerationResult (dict)
    if isinstance(raw, dict):
        return (
            raw.get("fit_markdown")       # version nettoyée/lisible (meilleure)
            or raw.get("raw_markdown")    # version brute complète
            or raw.get("markdown_with_citations")
            or ""
        )

    # crawl4ai < 0.3 ou réponse directe : markdown est déjà une string
    return raw or ""


def extraire_liens_articles(resultat: dict) -> list[str]:
    """
    Extrait les URLs d'articles depuis un résultat crawl4ai.
    Combine les liens internes détectés + les hrefs trouvés dans le markdown.

    Args:
        resultat: Dict résultat d'une page de listing crawl4ai.

    Returns:
        Liste d'URLs d'articles uniques et absolues.
    """
    liens_trouves: set[str] = set()

    # Source 1 : liens internes détectés automatiquement par crawl4ai
    liens_internes = resultat.get("links", {}).get("internal", [])
    for lien in liens_internes:
        href: str = lien.get("href", "")
        if _est_lien_article(href):
            liens_trouves.add(_normaliser_url(href))

    # Source 2 : liens extraits du markdown (fallback pour les pages SPA)
    markdown: str = _extraire_texte_markdown(resultat)   # ← fix : gérer dict ET str
    for ligne in markdown.splitlines():
        # Format markdown : [Titre](url)
        if "](http" in ligne or "](/" in ligne:
            debut = ligne.find("](") + 2
            fin   = ligne.find(")", debut)
            if debut > 1 and fin > debut:
                href = ligne[debut:fin]
                if _est_lien_article(href):
                    liens_trouves.add(_normaliser_url(href))

    return list(liens_trouves)


def _est_lien_article(href: str) -> bool:
    """Vérifie si l'URL correspond à un article d'actualité CNaPS."""
    if not href or href == CNAPS_ACTUALITES:
        return False
    # Exclure les pages de pagination, catégories, ancres, médias
    exclusions = ["/page/", "/categorie/", "#", ".jpg", ".png", ".pdf", "?"]
    if any(ex in href for ex in exclusions):
        return False
    # Inclure les patterns typiques d'articles CNaPS
    inclusions = ["/fr/", "cnaps.mg"]
    return any(inc in href for inc in inclusions) and len(href) > len(CNAPS_BASE_URL) + 5


def _normaliser_url(href: str) -> str:
    """Convertit les URLs relatives en absolues."""
    if href.startswith("http"):
        return href
    return CNAPS_BASE_URL + ("" if href.startswith("/") else "/") + href


def construire_urls_pagination(nb_pages: int) -> list[str]:
    """
    Génère les URLs de toutes les pages de pagination WordPress.
    Page 1 = URL de base, page N = /page/N/

    Args:
        nb_pages: Nombre total de pages à parcourir.

    Returns:
        Liste des URLs de pagination.
    """
    urls = [CNAPS_ACTUALITES]
    for num_page in range(2, nb_pages + 1):
        urls.append(f"{CNAPS_ACTUALITES}/page/{num_page}/")
    return urls


def decouvrir_nb_pages(resultat_page1: dict) -> int:
    """
    Détecte le nombre total de pages en cherchant les liens de pagination
    dans les liens internes ou le markdown.

    Args:
        resultat_page1: Résultat du crawl de la page 1.

    Returns:
        Nombre de pages détecté (minimum 1).
    """
    nb_pages_max = 1
    liens_internes = resultat_page1.get("links", {}).get("internal", [])

    for lien in liens_internes:
        href = lien.get("href", "")
        # Pattern WordPress : /categorie/actualites/page/N/
        if "/page/" in href and "/categorie/actualites" in href:
            try:
                partie_page = href.split("/page/")[-1].strip("/")
                num = int(partie_page.split("/")[0])
                nb_pages_max = max(nb_pages_max, num)
            except ValueError:
                continue

    return nb_pages_max


# ── Étape 3A : Crawl de la page 1 pour détecter la pagination ─────────────
print(f"🔍 Analyse de la page de listing : {CNAPS_ACTUALITES}")
resultats_p1 = crawl_urls([CNAPS_ACTUALITES])

if not resultats_p1:
    raise RuntimeError("❌ Impossible de crawler la page principale. Vérifiez le container Docker.")

resultat_page1 = resultats_p1[0]
nb_pages_total = decouvrir_nb_pages(resultat_page1)
print(f"📄 Nombre de pages de pagination détectées : {nb_pages_total}")

# ── Étape 3B : Crawl de toutes les pages de pagination ───────────────────────
urls_pagination = construire_urls_pagination(nb_pages_total)
urls_articles: list[str] = []

# Traiter la page 1 déjà crawlée
urls_articles.extend(extraire_liens_articles(resultat_page1))
print(f"   Page 1/{nb_pages_total} → {len(urls_articles)} article(s) trouvé(s)")

# Crawler les pages suivantes
if nb_pages_total > 1:
    for i, url_page in enumerate(tqdm(urls_pagination[1:], desc="📑 Pages de pagination"), start=2):
        time.sleep(DELAI_ENTRE_REQUETES)
        resultats = crawl_urls([url_page])
        if resultats:
            nouveaux = extraire_liens_articles(resultats[0])
            avant = len(urls_articles)
            for url in nouveaux:
                if url not in urls_articles:
                    urls_articles.append(url)
            print(f"   Page {i}/{nb_pages_total} → {len(urls_articles) - avant} nouveau(x) article(s)")

print(f"\n✅ Collecte terminée : {len(urls_articles)} articles uniques trouvés")
for url in urls_articles[:5]:
    print(f"   → {url}")

🔍 Analyse de la page de listing : https://www.cnaps.mg/fr/categorie/actualites
📄 Nombre de pages de pagination détectées : 1
   Page 1/1 → 40 article(s) trouvé(s)

✅ Collecte terminée : 40 articles uniques trouvés
   → https://www.cnaps.mg/fr/cnaps/historique
   → https://www.cnaps.mg/fr/video
   → https://www.cnaps.mg/fr/faq
   → https://www.cnaps.mg/fr/actualite/tontolo-andro-mampivoatra
   → https://www.cnaps.mg/media/cache/article/uploads/images/edf5434de73d7b51e51ccfdaa64f0eab.jpeg


## Cellule 4 — Scraping du contenu texte de chaque article

In [5]:
TAILLE_BATCH = 5  # Nombre d'articles crawlés en parallèle par appel API


def extraire_contenu_article(resultat: dict) -> dict:
    """
    Extrait le titre, la date et le texte complet d'un article depuis
    le résultat crawl4ai.

    Args:
        resultat: Dict résultat crawl4ai pour une page d'article.

    Returns:
        Dict avec les clés : url, titre, date, texte_complet, nb_mots.
    """
    markdown: str = _extraire_texte_markdown(resultat)   # ← fix : gérer dict ET str
    url: str      = resultat.get("url", "")

    # Titre = première ligne H1 du markdown
    titre = ""
    date  = ""
    for ligne in markdown.splitlines():
        ligne = ligne.strip()
        if ligne.startswith("# ") and not titre:
            titre = ligne[2:].strip()
        # Heuristique date : ligne courte contenant des chiffres et "/" ou "-"
        if not date and any(c.isdigit() for c in ligne):
            if ("/" in ligne or "-" in ligne) and len(ligne) < 30:
                date = ligne

    # Texte complet = markdown nettoyé (sans les balises # et liens images)
    lignes_texte = [
        l for l in markdown.splitlines()
        if l.strip()
        and not l.strip().startswith("![")   # exclure images
        and not l.strip().startswith("---")  # exclure séparateurs
    ]
    texte_complet = "\n".join(lignes_texte).strip()
    nb_mots = len(texte_complet.split())

    return {
        "url":           url,
        "titre":         titre or "(sans titre)",
        "date":          date  or "(date inconnue)",
        "texte_complet": texte_complet,
        "nb_mots":       nb_mots,
    }


# ── Scraping par batch ────────────────────────────────────────────────────────
if not urls_articles:
    print("⚠️  Aucun article à scraper. Exécutez d'abord la Cellule 3.")
else:
    print(f"⚡ Scraping de {len(urls_articles)} articles (batch de {TAILLE_BATCH})...\n")
    donnees_articles: list[dict] = []
    articles_echoues: list[str]  = []

    # Découper en batches pour éviter de surcharger le container
    batches = [
        urls_articles[i : i + TAILLE_BATCH]
        for i in range(0, len(urls_articles), TAILLE_BATCH)
    ]

    for idx, batch in enumerate(tqdm(batches, desc="🔄 Batches"), start=1):
        resultats = crawl_urls(batch, attendre_js=False)

        if not resultats:
            articles_echoues.extend(batch)
            continue

        for res in resultats:
            md = _extraire_texte_markdown(res)
            if res.get("success") or res.get("status") == "success" or md:
                article = extraire_contenu_article(res)
                donnees_articles.append(article)
                print(f"   ✅ [{article['nb_mots']} mots] {article['titre'][:60]}")
            else:
                url_echouee = res.get("url", "URL inconnue")
                articles_echoues.append(url_echouee)
                print(f"   ⚠️  Échec : {url_echouee}")

        # Pause entre les batches
        if idx < len(batches):
            time.sleep(DELAI_ENTRE_REQUETES)

    print(f"\n✅ Scraping terminé :")
    print(f"   • {len(donnees_articles)} articles extraits avec succès")
    print(f"   • {len(articles_echoues)} articles en échec")
    if articles_echoues:
        print("\n   URLs en échec :")
        for url in articles_echoues:
            print(f"      - {url}")

⚡ Scraping de 40 articles (batch de 5)...



ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## Cellule 5 — Sauvegarde CSV + affichage DataFrame

In [ ]:
FICHIER_CSV  = "cnaps_actualites_complet.csv"
FICHIER_JSON = "cnaps_actualites_complet.json"

if not donnees_articles:
    print("⚠️  Aucune donnée à sauvegarder. Exécutez d'abord la Cellule 4.")
else:
    df = pd.DataFrame(donnees_articles)

    # ── Sauvegarde CSV (aperçu court du texte pour lisibilité) ───────────────
    df_csv = df.copy()
    df_csv["apercu_texte"] = df_csv["texte_complet"].str[:400] + "..."
    df_csv.drop(columns=["texte_complet"], inplace=True)
    df_csv.to_csv(FICHIER_CSV, index=False, encoding="utf-8-sig")
    print(f"💾 CSV sauvegardé       → {FICHIER_CSV}")

    # ── Sauvegarde JSON (texte intégral pour ingestion RAG) ──────────────────
    df.to_json(FICHIER_JSON, orient="records", force_ascii=False, indent=2)
    print(f"💾 JSON sauvegardé      → {FICHIER_JSON}")

    # ── Statistiques ─────────────────────────────────────────────────────────
    print(f"\n📊 Statistiques :")
    print(f"   • Articles extraits  : {len(df)}")
    print(f"   • Mots total         : {df['nb_mots'].sum():,}")
    print(f"   • Mots moyen/article : {df['nb_mots'].mean():.0f}")
    print(f"   • Article le + long  : {df['nb_mots'].max()} mots")

    # ── Affichage aperçu dans Jupyter ────────────────────────────────────────
    pd.set_option("display.max_colwidth", 80)
    display(df[["titre", "date", "nb_mots", "url"]].sort_values("nb_mots", ascending=False))